# LeetCode 246 — Strobogrammatic Number bằng Python

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

- Link đề: https://leetcode.com/problems/strobogrammatic-number/
- Độ khó: **Easy** · Chủ đề: Hash Table, Two Pointers, String

Chữ ký hàm LeetCode:

```python
class Solution:
    def isStrobogrammatic(self, num: str) -> bool:

```

**Mục tiêu học**

1. Hiểu "số strobogrammatic": số **trông y hệt** khi xoay 180°.
2. Xác định bảng xoay: chữ số nào xoay được và xoay thành chữ số gì.
3. Ôn `dict` (tra cứu), `in`, slicing `[::-1]`, `reversed`, `"".join`.
4. Đi từ cách "xoay cả chuỗi rồi so sánh" tới **hai con trỏ** đối xứng — giống palindrome nhưng so qua bảng ánh xạ.
5. Mở rộng sang 247 (sinh mọi số dài `n`) và 248 (đếm trong khoảng).

## 1. Đề bài và ví dụ

Cho chuỗi `num` biểu diễn một số nguyên. Trả `True` nếu `num` là **strobogrammatic** — tức là khi xoay tờ giấy 180° (lộn ngược), số đọc được vẫn y như cũ.

| `num` | Kết quả | Vì sao |
|---|---|---|
| `"69"` | `True` | xoay 180°: `9` thành `6` ở đầu, `6` thành `9` ở cuối → `"69"` |
| `"88"` | `True` | `8` xoay vẫn là `8` |
| `"962"` | `False` | `2` xoay không thành chữ số hợp lệ |
| `"1"` | `True` | `1` xoay vẫn là `1` |
| `"818"` | `True` | đối xứng qua giữa, mọi chữ tự xoay |

**Constraints (theo hiểu biết về đề):**

- `1 <= num.length <= 50`
- `num` chỉ gồm chữ số
- `num` không có số 0 vô nghĩa ở đầu, trừ chính số `"0"`

## 2. Xoay 180° nghĩa là gì? — hai việc xảy ra cùng lúc

Khi xoay cả tờ giấy 180°:

1. **Thứ tự bị đảo:** chữ số cuối lên đầu, chữ số đầu xuống cuối.
2. **Mỗi chữ số tự xoay:** `6 → 9`, `9 → 6`, `0 → 0`, `1 → 1`, `8 → 8`. Các chữ `2, 3, 4, 5, 7` xoay **không** thành chữ số (ví dụ `7` lộn ngược trông như chữ `L`).

Bảng xoay:

| chữ số | 0 | 1 | 6 | 8 | 9 | 2, 3, 4, 5, 7 |
|---|---|---|---|---|---|---|
| sau khi xoay | 0 | 1 | 9 | 8 | 6 | (không hợp lệ) |

Ví dụ `"69"` từng bước: đảo thứ tự → `"96"`; xoay từng chữ: `9→6`, `6→9` → `"69"`. Giống ban đầu → `True`.

Ví dụ `"66"`: đảo → `"66"`; xoay → `"99"` ≠ `"66"` → `False`. (Đây là bẫy: `"66"` là palindrome nhưng **không** strobogrammatic.)

## 3. Nền tảng Python: `dict` làm bảng tra

In [1]:
rotate = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
print(rotate["6"], rotate["8"])
print("2" in rotate, "9" in rotate)     # kiểm tra key có trong dict không
print(rotate.get("7"))                  # key thiếu -> None (không lỗi)

9 8
False True
None


- `rotate["6"]` tra giá trị của key `"6"` → `"9"`. Nếu key không có → `KeyError`.
- `"2" in rotate` kiểm tra key, trung bình `O(1)`.
- `rotate.get("7")` trả `None` khi thiếu key — tiện để phát hiện chữ số không xoay được.

Lưu ý: key là **chuỗi** `"6"`, không phải số `6` — vì `num` là chuỗi, mỗi ký tự là chuỗi dài 1.

In [2]:
num = "69"
print(type(num[0]), repr(num[0]))
try:
    print(rotate[6])
except KeyError as e:
    print("KeyError khi dùng số nguyên 6 làm key:", e)

<class 'str'> '6'
KeyError khi dùng số nguyên 6 làm key: 6


### Đảo chuỗi: `[::-1]` và `reversed`

In [3]:
num = "1689"
print(num[::-1])                     # chuỗi đảo
print(list(reversed(num)))           # reversed trả iterator, đổi sang list để xem
print("".join(reversed(num)))        # nối lại thành chuỗi

9861
['9', '8', '6', '1']
9861


## 4. Cách 1 — Xoay cả chuỗi rồi so sánh (baseline)

Mô phỏng đúng định nghĩa:

1. Duyệt `num` **từ cuối về đầu** (đảo thứ tự).
2. Mỗi chữ số: nếu không có trong `rotate` → `False`; ngược lại lấy chữ đã xoay.
3. Nối thành chuỗi `rotated`, so với `num`.

In [4]:
def is_strobo_build(num: str) -> bool:
    rotated = []
    for ch in reversed(num):
        if ch not in rotate:
            return False
        rotated.append(rotate[ch])
    return "".join(rotated) == num

for x in ["69", "88", "962", "1", "818", "66", "10", "609"]:
    print(f"{x:>4} -> {is_strobo_build(x)}")

  69 -> True
  88 -> True
 962 -> False
   1 -> True
 818 -> True
  66 -> False
  10 -> False
 609 -> True


Cùng ý tưởng viết gọn bằng **generator expression** và `all`:

In [5]:
def is_strobo_build_short(num: str) -> bool:
    if not all(ch in rotate for ch in num):
        return False
    return "".join(rotate[ch] for ch in reversed(num)) == num

print([is_strobo_build_short(x) for x in ["69", "88", "962", "66"]])

[True, True, False, False]


`"".join(rotate[ch] for ch in reversed(num))` — biểu thức trong ngoặc tạo từng chữ đã xoay "theo yêu cầu", không cần list trung gian.

### Nút thắt

- Cách 1 tạo chuỗi mới dài `n` → `O(n)` bộ nhớ phụ, và luôn xoay **hết** chuỗi rồi mới so sánh (không dừng sớm khi lệch ngay cặp đầu).
- Nhận xét: so `rotated == num` tức là so `rotate[num[n-1-i]] == num[i]` cho mọi `i`. Mỗi cặp vị trí đối xứng `(i, n-1-i)` được kiểm tra **hai lần** (một lần ở `i`, một lần ở `n-1-i`). Chỉ cần kiểm tra nửa chuỗi!

## 5. Insight — hai con trỏ đối xứng như palindrome

Gọi `left = 0`, `right = n - 1`. Sau khi xoay, chữ ở vị trí `left` của số mới chính là `rotate[num[right]]`. Muốn số không đổi:

```text
rotate[num[right]] == num[left]     (với mọi cặp đối xứng)
```

Điều kiện này **đối xứng**: vì bảng xoay là song ánh (`6↔9`, các chữ khác tự ánh xạ), `rotate[a] == b` ⇔ `rotate[b] == a`. Nên kiểm tra một chiều là đủ.

Khi `left == right` (chữ ở giữa của chuỗi độ dài lẻ): chữ đó phải tự xoay thành chính nó → chỉ được `0`, `1`, `8`. Công thức trên tự lo việc này: `rotate["6"] == "6"`? Không → `False`.

In [6]:
for a, b in [("6", "9"), ("9", "6"), ("8", "8"), ("6", "6"), ("1", "8")]:
    print(f"cặp (trái={a}, phải={b}): rotate[phải] = {rotate.get(b)} -> hợp lệ? {rotate.get(b) == a}")

cặp (trái=6, phải=9): rotate[phải] = 6 -> hợp lệ? True
cặp (trái=9, phải=6): rotate[phải] = 9 -> hợp lệ? True
cặp (trái=8, phải=8): rotate[phải] = 8 -> hợp lệ? True
cặp (trái=6, phải=6): rotate[phải] = 9 -> hợp lệ? False
cặp (trái=1, phải=8): rotate[phải] = 8 -> hợp lệ? False


## 6. Lời giải chính — hai con trỏ + bảng xoay

1. `left, right = 0, len(num) - 1`.
2. Trong khi `left <= right`:
   - lấy `r = rotate.get(num[right])`; nếu `r is None` (chữ không xoay được) hoặc `r != num[left]` → `False`;
   - `left += 1`, `right -= 1`.
3. Hết vòng → `True`.

Dùng `<=` (không phải `<`) để **chữ ở giữa** cũng được kiểm tra.

In [7]:
def is_strobo_two_pointers(num: str) -> bool:
    left, right = 0, len(num) - 1
    while left <= right:
        if rotate.get(num[right]) != num[left]:
            return False
        left += 1
        right -= 1
    return True

for x in ["69", "88", "962", "1", "818", "66", "6", "609", "906"]:
    print(f"{x:>4} -> {is_strobo_two_pointers(x)}")

  69 -> True
  88 -> True
 962 -> False
   1 -> True
 818 -> True
  66 -> False
   6 -> False
 609 -> True
 906 -> True


Mẹo nhỏ: `rotate.get(num[right]) != num[left]` gộp luôn trường hợp chữ không xoay được, vì `None != num[left]` luôn đúng (`num[left]` là chuỗi). Gọn mà vẫn đúng.

## 7. Trace từng bước

`num = "16891"`:

| left | right | `num[left]` | `num[right]` | `rotate[num[right]]` | Khớp? |
|---|---|---|---|---|---|
| 0 | 4 | 1 | 1 | 1 | ✓ |
| 1 | 3 | 6 | 9 | 6 | ✓ |
| 2 | 2 | 8 | 8 | 8 | ✓ (chữ giữa) |
| 3 | 1 | – | – | – | dừng (`left > right`) → `True` |

In [8]:
def trace_strobo(num: str) -> bool:
    print(f"num = {num!r}")
    left, right = 0, len(num) - 1
    while left <= right:
        a, b = num[left], num[right]
        r = rotate.get(b)
        tag = " (chữ giữa)" if left == right else ""
        print(f"  left={left}, right={right}: num[left]={a}, num[right]={b}, rotate[{b}]={r}{tag}", end="")
        if r != a:
            print("  -> KHÔNG khớp -> False")
            return False
        print("  -> khớp")
        left += 1
        right -= 1
    print("  hết cặp -> True")
    return True

trace_strobo("16891")
trace_strobo("6")
trace_strobo("6229")

num = '16891'
  left=0, right=4: num[left]=1, num[right]=1, rotate[1]=1  -> khớp
  left=1, right=3: num[left]=6, num[right]=9, rotate[9]=6  -> khớp
  left=2, right=2: num[left]=8, num[right]=8, rotate[8]=8 (chữ giữa)  -> khớp
  hết cặp -> True
num = '6'
  left=0, right=0: num[left]=6, num[right]=6, rotate[6]=9 (chữ giữa)  -> KHÔNG khớp -> False
num = '6229'
  left=0, right=3: num[left]=6, num[right]=9, rotate[9]=6  -> khớp
  left=1, right=2: num[left]=2, num[right]=2, rotate[2]=None  -> KHÔNG khớp -> False


False

`print(..., end="")` in không xuống dòng, để dòng tiếp theo in tiếp trên cùng dòng.

## 8. Cách 3 — Tập các **cặp hợp lệ**

Thay vì dict, liệt kê thẳng mọi cặp `(trái, phải)` hợp lệ: `{"00", "11", "88", "69", "96"}`. Mỗi bước ghép `num[left] + num[right]` rồi kiểm tra có trong tập không.

In [9]:
VALID_PAIRS = {"00", "11", "88", "69", "96"}

def is_strobo_pairs(num: str) -> bool:
    left, right = 0, len(num) - 1
    while left <= right:
        if num[left] + num[right] not in VALID_PAIRS:
            return False
        left += 1
        right -= 1
    return True

print([is_strobo_pairs(x) for x in ["69", "88", "962", "1", "818", "66", "6"]])

[True, True, False, True, True, False, False]


Khi `left == right`, chuỗi ghép là `"88"`, `"66"`... — chỉ `"00"`, `"11"`, `"88"` có trong tập → chữ giữa đúng là phải tự đối xứng. Không cần xử lý riêng!

`set` (tập hợp) giống dict nhưng chỉ có key; `x in set` trung bình `O(1)`.

## 9. Cách 4 — So sánh với `str.translate` (một dòng)

Python có `str.maketrans` tạo bảng thay ký tự, `str.translate` thay hàng loạt. Ký tự cần **xóa** truyền ở tham số thứ ba.

In [10]:
TABLE = str.maketrans("01689", "01986", "23457")

def is_strobo_translate(num: str) -> bool:
    rotated = num[::-1].translate(TABLE)
    return len(rotated) == len(num) and rotated == num

print("1689".translate(TABLE), "| '962' sau translate:", repr("962"[::-1].translate(TABLE)))
print([is_strobo_translate(x) for x in ["69", "88", "962", "1", "818", "66"]])

1986 | '962' sau translate: '96'
[True, True, False, True, True, False]


Chữ số không xoay được bị **xóa** khi translate → chuỗi ngắn đi → so `len` (thực ra `rotated == num` đã đủ vì độ dài khác thì chuỗi khác, nhưng ghi rõ cho dễ hiểu). Cách này gọn nhưng "ma thuật"; trong phỏng vấn nên trình bày cách hai con trỏ.

## 10. Code nộp LeetCode

In [11]:
class Solution:
    def isStrobogrammatic(self, num: str) -> bool:
        rotate = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
        left, right = 0, len(num) - 1
        while left <= right:
            if rotate.get(num[right]) != num[left]:
                return False
            left += 1
            right -= 1
        return True

sol = Solution()
print(sol.isStrobogrammatic("69"), sol.isStrobogrammatic("88"), sol.isStrobogrammatic("962"))

True True False


## 11. Vì sao đúng? + Độ phức tạp

Gọi `n = len(num)`. Số sau khi xoay là `R` với `R[i] = rotate(num[n-1-i])`. `num` strobogrammatic ⇔ mọi chữ xoay được và `R[i] == num[i]` với mọi `i`.

- Với cặp `(i, n-1-i)`, `i < n-1-i`: điều kiện ở `i` là `rotate(num[n-1-i]) == num[i]`; điều kiện ở `n-1-i` là `rotate(num[i]) == num[n-1-i]`. Hai điều kiện **tương đương** vì `rotate` là song ánh tự nghịch đảo (`rotate(rotate(x)) = x`). → chỉ cần xét `i ≤ n-1-i`, đúng là vòng `left <= right`.
- `rotate.get(...)` trả `None` cho chữ không xoay được → bị loại, đảm bảo điều kiện "mọi chữ xoay được" (mọi chữ đều nằm trong một cặp nào đó được xét).

**Invariant:** trước mỗi vòng, mọi cặp `(i, n-1-i)` với `i < left` đều hợp lệ.

| Cách | Time | Space phụ |
|---|---|---|
| Xoay cả chuỗi rồi so | `O(n)` | `O(n)` |
| Hai con trỏ + dict | `O(n)` (≈ `n/2` vòng, dừng sớm) | `O(1)` (dict 5 phần tử cố định) |
| Hai con trỏ + tập cặp | `O(n)` | `O(1)` |
| `translate` | `O(n)` | `O(n)` |

## 12. Bộ test

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}")

sol = Solution()
check("69", sol.isStrobogrammatic("69"), True)
check("88", sol.isStrobogrammatic("88"), True)
check("962", sol.isStrobogrammatic("962"), False)
check("1", sol.isStrobogrammatic("1"), True)
check("0", sol.isStrobogrammatic("0"), True)
check("6 (chữ giữa không tự đối xứng)", sol.isStrobogrammatic("6"), False)
check("66 (palindrome nhưng không strobo)", sol.isStrobogrammatic("66"), False)
check("101", sol.isStrobogrammatic("101"), True)
check("609 (giữa là 0)", sol.isStrobogrammatic("609"), True)
check("2", sol.isStrobogrammatic("2"), False)
check("dài 50: '6969...69'", sol.isStrobogrammatic("69" * 25), True)
check("dài 50: '66...699...9'", sol.isStrobogrammatic("6" * 25 + "9" * 25), True)
check("dài 50 lệch một chữ", sol.isStrobogrammatic("6" * 25 + "9" * 24 + "8"), False)

PASS  69
PASS  88
PASS  962
PASS  1
PASS  0
PASS  6 (chữ giữa không tự đối xứng)
PASS  66 (palindrome nhưng không strobo)
PASS  101
PASS  609 (giữa là 0)
PASS  2
PASS  dài 50: '6969...69'
PASS  dài 50: '66...699...9'
PASS  dài 50 lệch một chữ


Có thể bạn đoán `"69" * 25 = "6969...69"` là `False` vì nó "không đối xứng". Nhưng hãy xét từng cặp: `(s[0], s[49]) = (6, 9)` ✓, `(s[1], s[48]) = (9, 6)` ✓, ... mọi cặp đều hợp lệ → `True`. Ta kiểm chứng bằng baseline "xoay cả chuỗi" (cài đặt bám sát định nghĩa, dùng làm **oracle**):

In [13]:
s = "69" * 25
print("baseline:", is_strobo_build(s), "| two pointers:", sol.isStrobogrammatic(s))
print("cặp (0,49):", s[0], s[49], "| cặp (1,48):", s[1], s[48])

baseline: True | two pointers: True
cặp (0,49): 6 9 | cặp (1,48): 9 6


**Bài học:** expected trong test do ta tự đoán cũng có thể sai. Hãy đối chiếu với một cài đặt bám sát định nghĩa trên nhiều input ngẫu nhiên:

In [14]:
import random
rng = random.Random(246)
solvers = [is_strobo_build, is_strobo_build_short, is_strobo_two_pointers,
           is_strobo_pairs, is_strobo_translate, Solution().isStrobogrammatic]
count_true = 0
for _ in range(5000):
    n = rng.randint(1, 8)
    if rng.random() < 0.5:     # sinh số chắc chắn strobo
        half = [rng.choice("01689") for _ in range(n // 2)]
        mid = [rng.choice("018")] if n % 2 else []
        x = "".join(half + mid + [rotate[c] for c in reversed(half)])
    else:
        x = "".join(rng.choice("0123456789") for _ in range(n))
    outs = [f(x) for f in solvers]
    assert len(set(outs)) == 1, (x, outs)
    count_true += outs[0]
print(f"5000 chuỗi: 6 cài đặt khớp nhau — PASS ({count_true} chuỗi strobogrammatic)")

5000 chuỗi: 6 cài đặt khớp nhau — PASS (2637 chuỗi strobogrammatic)


## 13. Lỗi người mới hay gặp

### Lỗi 1: Coi strobogrammatic là palindrome

In [15]:
def wrong_palindrome(num):
    return num == num[::-1] and all(c in "01689" for c in num)

print("66 ->", wrong_palindrome("66"), "(phải False) | 69 ->", wrong_palindrome("69"), "(phải True)")

66 -> True (phải False) | 69 -> False (phải True)


**Vì sao sai:** palindrome chỉ đảo thứ tự, còn xoay 180° **vừa đảo vừa biến đổi** từng chữ (`6↔9`). **Sửa:** so qua bảng `rotate`.

### Lỗi 2: Quên kiểm tra chữ ở giữa (`left < right` thay vì `<=`)

In [16]:
def wrong_strict(num):
    left, right = 0, len(num) - 1
    while left < right:
        if rotate.get(num[right]) != num[left]:
            return False
        left += 1
        right -= 1
    return True

print("'6' ->", wrong_strict("6"), "(phải False) | '161' ->", wrong_strict("161"), "(phải False)")

'6' -> True (phải False) | '161' -> True (phải False)


**Vì sao sai:** với độ dài lẻ, chữ giữa không được xét; `6` hoặc `9` ở giữa xoay thành chữ khác. **Sửa:** `while left <= right`.

### Lỗi 3: Chỉ kiểm tra "mọi chữ thuộc {0,1,6,8,9}"
`"16"` toàn chữ xoay được, nhưng xoay thành `"91"` ≠ `"16"`. Phải kiểm tra **từng cặp đối xứng**.

### Lỗi 4: Quên `2,3,4,5,7` → `KeyError`

In [17]:
def wrong_keyerror(num):
    return "".join(rotate[c] for c in reversed(num)) == num

try:
    wrong_keyerror("25")
except KeyError as e:
    print("KeyError:", e, "-> dùng rotate.get hoặc kiểm tra `in` trước")

KeyError: '5' -> dùng rotate.get hoặc kiểm tra `in` trước


## 14. Mẹo nhận dạng pattern

- "Đọc ngược / đối xứng / xoay / lật" một chuỗi → **hai con trỏ từ hai đầu**.
- Khi hai đầu không cần **bằng nhau** mà cần **tương ứng** theo một quy tắc → **bảng ánh xạ (dict)** hoặc **tập cặp hợp lệ**.
- Độ dài lẻ → luôn tự hỏi "phần tử giữa xử lý thế nào?" (`<=` hay `<`).

## 15. Biến thể

### 247 — Strobogrammatic Number II: sinh mọi số strobogrammatic dài `n`

Xây từ trong ra ngoài: số dài `n` = cặp `(a, b)` bọc quanh số dài `n - 2`. Không được dùng cặp `"00"` ở vòng ngoài cùng (trừ khi `n == 1`) vì tạo số 0 ở đầu.

In [18]:
def find_strobogrammatic(n: int) -> list[str]:
    pairs = [("0", "0"), ("1", "1"), ("6", "9"), ("8", "8"), ("9", "6")]

    def build(m: int) -> list[str]:
        if m == 0:
            return [""]
        if m == 1:
            return ["0", "1", "8"]
        inner = build(m - 2)
        out = []
        for s in inner:
            for a, b in pairs:
                if m == n and a == "0":     # vòng ngoài cùng không được bắt đầu bằng 0
                    continue
                out.append(a + s + b)
        return out

    return build(n)

print(find_strobogrammatic(1))
print(find_strobogrammatic(2))
print(find_strobogrammatic(3))

['0', '1', '8']
['11', '69', '88', '96']
['101', '609', '808', '906', '111', '619', '818', '916', '181', '689', '888', '986']


### 248 — Strobogrammatic Number III (Hard): đếm số strobogrammatic trong khoảng `[low, high]`
Sinh theo mọi độ dài từ `len(low)` đến `len(high)` rồi lọc theo giá trị — dùng lại `find_strobogrammatic`.

## 16. Bài tập tự luyện (TODO)

### Bài 1 — Tính tay
Các số sau có strobogrammatic không: `"808"`, `"619"`, `"9006"`, `"6009"`, `"11"`, `"181"`? Ghi từng cặp `(left, right)`.

### Bài 2 — Xoay một số (trả chuỗi sau khi xoay, hoặc `None` nếu không xoay được)

In [19]:
def exercise_rotate(num: str):
    # TODO: duyệt reversed(num), tra rotate, gặp chữ lạ trả None
    pass

# assert exercise_rotate("1689") == "6891"
# assert exercise_rotate("25") is None

### Bài 3 — Tự viết lời giải hai con trỏ

In [20]:
def exercise_is_strobo(num: str) -> bool:
    # TODO: left, right; while left <= right; so rotate.get(num[right]) với num[left]
    pass

# assert exercise_is_strobo("69") == True
# assert exercise_is_strobo("6") == False

### Bài 4 — Số "khó hiểu" (Confusing Number, LeetCode 1056)
Một số là *confusing* nếu xoay 180° được một **số hợp lệ khác** với ban đầu (bỏ qua số 0 ở đầu sau khi xoay). Ví dụ `6 → 9` (True), `89 → 68` (True), `11 → 11` (False), `25` (False). Input là `int`.

In [21]:
def exercise_confusing(n: int) -> bool:
    # TODO: xoay bằng chuỗi hoặc bằng % 10, // 10; so sánh với n
    pass

# assert exercise_confusing(6) == True
# assert exercise_confusing(89) == True
# assert exercise_confusing(11) == False
# assert exercise_confusing(25) == False

### Bài 5 — Đếm số strobogrammatic từ 0 đến `N`
Viết hàm đếm (brute force, `N <= 10^4`).

In [22]:
def exercise_count_strobo(N: int) -> int:
    # TODO: duyệt 0..N, dùng hàm kiểm tra
    pass

# assert exercise_count_strobo(100) == 7   # 0, 1, 8, 11, 69, 88, 96

### Bài 6 — Mini challenge: Strobogrammatic III
Đếm số strobogrammatic `x` với `low <= x <= high` (chuỗi, không số 0 đầu). Ví dụ `low="50", high="100"` → `3` (69, 88, 96).

In [23]:
def exercise_strobo_in_range(low: str, high: str) -> int:
    # TODO: sinh theo độ dài bằng find_strobogrammatic, lọc theo int
    pass

# assert exercise_strobo_in_range("50", "100") == 3
# assert exercise_strobo_in_range("0", "0") == 1

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Áp bảng xoay cho từng cặp | `"9006"`: cặp (9,6) ✓, (0,0) ✓ | `"619"`: chữ giữa `1` ✓, cặp (6,9) ✓ |
| 2 | Mục 4 | `if ch not in rotate: return None` | `"".join(...)` |
| 3 | Mục 6 | nhớ `<=` | `rotate.get` tránh KeyError |
| 4 | Xoay được rồi so khác | `int(rotated) != n` | xoay bằng chuỗi `str(n)` |
| 5 | Dùng lại `Solution` | `sum(1 for x in range(N+1) if ...)` | đổi `x` sang `str` |
| 6 | Sinh theo từng độ dài | `range(len(low), len(high) + 1)` | lọc `int(low) <= int(x) <= int(high)` |

## 18. Đáp án tham khảo, template và checklist

Bài 1: `"808"` ✓, `"619"` ✓, `"9006"` ✓, `"6009"` ✓, `"11"` ✓, `"181"` ✓ — tất cả `True`. (Thử thêm `"916"`: cặp (9, 6) ✓ nhưng giữa là `1` ✓ → cũng `True`!)

In [24]:
def answer_rotate(num):
    out = []
    for ch in reversed(num):
        if ch not in rotate:
            return None
        out.append(rotate[ch])
    return "".join(out)

def answer_is_strobo(num):
    left, right = 0, len(num) - 1
    while left <= right:
        if rotate.get(num[right]) != num[left]:
            return False
        left, right = left + 1, right - 1
    return True

def answer_confusing(n):
    r = answer_rotate(str(n))
    return r is not None and int(r) != n

def answer_count_strobo(N):
    return sum(1 for x in range(N + 1) if answer_is_strobo(str(x)))

def answer_strobo_in_range(low, high):
    total = 0
    for length in range(len(low), len(high) + 1):
        for x in find_strobogrammatic(length):
            if int(low) <= int(x) <= int(high):
                total += 1
    return total

print([answer_is_strobo(x) for x in ["808", "619", "9006", "6009", "11", "181", "916"]])
assert answer_rotate("1689") == "6891" and answer_rotate("25") is None
assert answer_is_strobo("69") and not answer_is_strobo("6")
assert answer_confusing(6) and answer_confusing(89) and not answer_confusing(11) and not answer_confusing(25)
assert answer_confusing(10)            # 10 -> "01" = 1 khác 10
assert answer_count_strobo(100) == 7
assert answer_strobo_in_range("50", "100") == 3 and answer_strobo_in_range("0", "0") == 1
assert answer_strobo_in_range("0", "9999") == answer_count_strobo(9999)
for n in range(1, 6):
    gen = set(find_strobogrammatic(n))
    brute = {str(x) for x in range(10 ** (n - 1) if n > 1 else 0, 10 ** n) if answer_is_strobo(str(x))}
    assert gen == brute, n
print("Đáp án tham khảo: tất cả test PASS")

[True, True, True, True, True, True, True]
Đáp án tham khảo: tất cả test PASS


### Template nhớ nhanh

```python
rotate = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
left, right = 0, len(num) - 1
while left <= right:
    if rotate.get(num[right]) != num[left]:
        return False
    left += 1; right -= 1
return True
```

### Checklist submit

- [ ] Bảng xoay đủ 5 chữ: `0,1,6,8,9` (và `6↔9`).
- [ ] So **qua bảng xoay**, không so bằng nhau như palindrome.
- [ ] `while left <= right` để xét chữ ở giữa.
- [ ] Chữ không xoay được (`2,3,4,5,7`) bị loại, không gây `KeyError`.
- [ ] Key của dict là **chuỗi** `"6"`, không phải số `6`.